In [11]:
# 05_data_cleaning.ipynb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer
import warnings
warnings.filterwarnings('ignore')

# Cargar datos crudos
DATA_PATH = "../data/raw/dataset_pucp.csv"
df = pd.read_csv(DATA_PATH)
print(f"Dataset original: {df.shape}")

Dataset original: (12330, 18)


In [16]:
# Ver duplicados
duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {duplicados}")

# Eliminar duplicados
df_clean = df.drop_duplicates()
print(f"Dataset sin duplicados: {df_clean.shape}")

Filas duplicadas: 125
Dataset sin duplicados: (12205, 18)


In [17]:
# Ver nulos
print(df_clean.isnull().sum())

# Imputar según el tipo de variable
# Ejemplo: numéricas con mediana
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())

# Categóricas con moda
categorical_cols = df_clean.select_dtypes(include=['object']).columns
df_clean[categorical_cols] = df_clean[categorical_cols].fillna(df_clean[categorical_cols].mode().iloc[0])

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64


In [18]:
# Usar IQR para detectar outliers
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower) | (data[column] > upper)]
    return len(outliers)

# Winsorizing (limitar valores extremos)
from scipy.stats.mstats import winsorize

for col in numeric_cols:
    df_clean[col] = winsorize(df_clean[col], limits=[0.05, 0.05])

In [6]:
# Ver tipos actuales
print(df_clean.dtypes)

# Convertir columnas que deberían ser categóricas
categorical_columns = ['Month', 'VisitorType', 'Weekend', 'Revenue']
for col in categorical_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('category')

Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object


In [19]:
# OneHot Encoding para variables nominales
df_encoded = pd.get_dummies(df_clean, columns=['Month', 'VisitorType'], drop_first=True)

# Label Encoding para target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_encoded['Revenue'] = le.fit_transform(df_encoded['Revenue'])

In [20]:
# Seleccionar numéricas (excepto target)
numeric_features = df_encoded.select_dtypes(include=[np.number]).columns
numeric_features = [col for col in numeric_features if col != 'Revenue']

# Aplicar StandardScaler
scaler = StandardScaler()
df_encoded[numeric_features] = scaler.fit_transform(df_encoded[numeric_features])

In [9]:
# Guardar en data/interim/
df_encoded.to_csv("../data/interim/dataset_clean.csv", index=False)
print(f"✅ Dataset limpio guardado: {df_encoded.shape}")

OSError: Cannot save file into a non-existent directory: '..\data\interim'

In [21]:
print("="*60)
print("VERIFICACIÓN DEL DATASET LIMPIO")
print("="*60)
print(f"Shape: {df_encoded.shape}")
print(f"Nulos: {df_encoded.isnull().sum().sum()}")
print(f"Duplicados: {df_encoded.duplicated().sum()}")
print(f"Tipos:\n{df_encoded.dtypes.value_counts()}")

VERIFICACIÓN DEL DATASET LIMPIO
Shape: (12205, 27)
Nulos: 0
Duplicados: 42
Tipos:
float64    14
bool       12
int64       1
Name: count, dtype: int64
